<a href="https://colab.research.google.com/github/oliveirasamuel5959/VRDFormer_VRD/blob/main/notebooks/colab_kaggle_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VRDFormer Training on Colab T4 with Kaggle Data

This notebook trains the full VRDFormer pipeline (Stage 1 + Stage 2) on VidOR using a Colab T4 GPU.
Data is hosted on Kaggle as a dataset — no Google Drive space needed.

**Prerequisites:**
- GitHub fork of VRDFormer_VRD with Kaggle configs + bug fixes
- Kaggle dataset uploaded (contains data/vidor/ + metadata + DETR weights)
- Kaggle API key (kaggle.json) from kaggle.com/settings → API → Create New Token

https://colab.research.google.com/github/oliveirasamuel5959/VRDFormer_VRD/blob/main/notebooks/colab_kaggle_train.ipynb

## 1. Clone Repo & Install Dependencies

In [1]:
# Clone your fork (replace YOUR_USERNAME)
!git clone https://github.com/oliveirasamuel5959/VRDFormer_VRD.git
%cd VRDFormer_VRD

# Install dependencies (PyTorch + torchvision pre-installed on Colab T4)
!pip install decord timm scipy lap -q
!pip install -U 'git+https://github.com/timmeinhardt/cocoapi.git#subdirectory=PythonAPI' -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Cloning into 'VRDFormer_VRD'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 158 (delta 51), reused 131 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 178.14 KiB | 11.88 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/VRDFormer_VRD
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
PyTorch: 2.11.0+cu128
CUDA available: True


## 2. Download Kaggle Dataset

In [2]:
# Upload your kaggle.json API key
from google.colab import files
files.upload()  # Select kaggle.json from your local machine

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

# Make data directory with metadata and weights
!mkdir -p data/metadata data/weights data/ckpts

# Download and extract (replace YOUR_KAGGLE_USERNAME)
!kaggle datasets download samuelpatricio/vrdformer-vidor
!unzip -q vrdformer-vidor.zip -d data
!rm vrdformer-vidor.zip



Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/samuelpatricio/vrdformer-vidor
License(s): unknown
100% 13.2G/13.2G [10:49<00:00, 21.8MB/s]

metadata already in place
Dataset ready.


In [3]:
# Move bundled metadata and weights into place
!mv data/vidor/metadata_bundle/* data/metadata/ 2>/dev/null || echo 'metadata already in place'
!mv data/vidor/detr-r101-2c7b67e5.pth data/weights/ 2>/dev/null || echo 'weights already in place'

print('Dataset ready.')

weights already in place
Dataset ready.


## 3. Verify GPU & Data

In [4]:
!nvidia-smi

import os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

checks = [
    ('data/vidor/videos', 'dir'),
    ('data/vidor/annotations/train', 'dir'),
    ('data/vidor/annotations/val', 'dir'),
    ('data/metadata/vidor_train_frames_stage1.json', 'file'),
    ('data/metadata/vidor_val_frames.json', 'file'),
    ('data/weights/detr-r101-2c7b67e5.pth', 'file'),
    ('configs/vidor_kaggle_stage1.json', 'file'),
    ('configs/vidor_kaggle_stage2.json', 'file'),
]

all_ok = True
for path, kind in checks:
    if kind == 'dir':
        exists = os.path.isdir(path)
    else:
        exists = os.path.isfile(path)
    size = ''
    if exists and kind == 'file':
        size = f' ({os.path.getsize(path) / 1e6:.1f} MB)'
    print(f'  {"✅" if exists else "❌"} {path}{size}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll checks passed — ready to train!')
else:
    print('\n⚠️  Some files missing — check the Kaggle dataset download.')

Fri Jul 31 12:38:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. Smoke Test (Optional)

Run 1 epoch on 100 videos to verify the pipeline works before committing 5+ hours to full training.

In [6]:
# Smoke test — uncomment to run (~20 min)
!python main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidorpart_local_stage1.json \
    --epochs 1
print('Smoke test skipped — uncomment the cell above to run it.')

Not using distributed mode
git:
  sha: 38b1c0db94bbef46a1260f2354f042484fa43250, status: clean, branch: main

Namespace(lr=5e-05, lr_backbone=1e-05, lr_drop=4, weight_decay=0.0001, batch_size=4, epochs=5, optimizer='adam', clip_max_norm=0.1, accumulate_steps=1.0, eval_skip=1, debug=True, vis_and_log_interval=10, schedule='linear_with_warmup', ema=False, ema_decay=0.9998, fraction_warmup_steps=0.01, frozen_weights=None, roi_pool_type='avg', backbone='resnet101', dilation=False, position_embedding='sine_3d_v2', enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8, num_queries=200, pre_norm=False, aux_loss=False, eval=False, eval_mode='evalGT', output_dir='data/ckpts/vidorpart_stage1', device='cuda', seed=42, resume='', resume_shift_neuron=False, pretrain='data/weights/detr-r101-2c7b67e5.pth', start_epoch=0, num_workers=0, dataset_config='configs/vidorpart_local_stage1.json', world_size=1, dist_url='env://', local_rank=-1, stage=1, coco_path='', vidvrd_p

## 5. Stage 1 Training

Pair detection + tracking. **~4-6 hours**, uses ~13-15 GB VRAM.

**What to watch:**
- Loss: ~2.5 → ~0.9 over 5 epochs
- sub_class_acc: ~30% → ~88%
- Checkpoints saved to `data/ckpts/vidor_stage1/`
- *Known issue:* NameError at epoch end is harmless (Stage 1 has no eval path)

In [ ]:
%%time
!python -m torch.distributed.launch \
    --master_port 47749 \
    --nproc_per_node=1 \
    main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_kaggle_stage1.json

### Verify Stage 1 Output

In [ ]:
import os, glob
ckpts = sorted(glob.glob('data/ckpts/vidor_stage1/checkpoint*.pth'))
print(f'Stage 1 checkpoints: {len(ckpts)}')
for ckpt in ckpts:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {ckpt:<55} {size_mb:.1f} MB')

## 6. Stage 2 Training

Temporal relation classification. **~1-2 hours**, uses ~10-12 GB VRAM.

Loads `checkpoint0004.pth` from Stage 1. Uses GT boxes via ROI Align (no box regression).

In [ ]:
%%time
!python -m torch.distributed.launch \
    --master_port 47745 \
    --nproc_per_node=1 \
    main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_kaggle_stage2.json

## 7. Evaluation

Evaluates on VidOR validation set. Reports detection mAP, recall@50/100, and tagging precision@1/5/10 — each under overall, zero-shot, and generalized-zero-shot settings.

In [ ]:
!python -m torch.distributed.launch \
    --nproc_per_node=1 \
    main.py \
    --eval \
    --dataset_config configs/vidor_kaggle_stage2.json \
    --resume data/ckpts/vidor_stage2/checkpoint.pth

## 8. Download Checkpoints

In [ ]:
# Zip and download checkpoints to your local machine
!zip -r checkpoints.zip data/ckpts/
from google.colab import files
files.download('checkpoints.zip')

print('\nDone! Key files:')
print('  Stage 1: data/ckpts/vidor_stage1/checkpoint0004.pth')
print('  Stage 2: data/ckpts/vidor_stage2/checkpoint.pth')

## Colab Keep-Alive

Colab disconnects after ~90 min of inactivity. Paste this in the browser console (F12) to prevent disconnects:

```javascript
function ClickConnect(){
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```